In [ ]:
# Install:
# pip install azure-storage-blob

from azure.storage.blob import BlobServiceClient

# Azure Storage Account
account_name = "insurancedataset2026"
account_key = "YOUR_ACCOUNT_KEY"

connection_string = (
    f"DefaultEndpointsProtocol=https;"
    f"AccountName={account_name};"
    f"AccountKey={account_key};"
    f"EndpointSuffix=core.windows.net"
)

# Container names
source_container = "source"
destination_container = "destination"

# Connect to Azure Blob Storage
blob_service = BlobServiceClient.from_connection_string(
    connection_string
)

source_client = blob_service.get_container_client(
    source_container
)

destination_client = blob_service.get_container_client(
    destination_container
)

# Move all Excel files
for blob in source_client.list_blobs():

    if blob.name.lower().endswith((".xlsx", ".xls")):

        source_blob = source_client.get_blob_client(blob.name)

        destination_blob = destination_client.get_blob_client(
            blob.name
        )

        # Download Excel file
        data = source_blob.download_blob().readall()

        # Upload to destination container
        destination_blob.upload_blob(
            data,
            overwrite=True
        )

        # Verify destination file
        uploaded = destination_blob.get_blob_properties()

        if uploaded.size == len(data):

            # Delete from source container
            source_blob.delete_blob()

            print("Moved:", blob.name)

        else:
            print("Failed:", blob.name)

print("All Excel files moved successfully!")

In [0]:
# ============================================================
# MOVE EXCEL FILES FROM SOURCE TO DESTINATION USING PYSPARK
# AZURE DATABRICKS + AZURE BLOB STORAGE
# ============================================================

# STEP 1: Storage Account Details

storage_account = "insurancedataset2026"
account_key = "YOUR_NEW_ACCOUNT_KEY"

source_container = "source"
destination_container = "destination"


# STEP 2: Configure Spark to Access Azure Blob Storage

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    account_key
)


# STEP 3: Define Source and Destination Paths

source_path = (
    f"wasbs://{source_container}@"
    f"{storage_account}.blob.core.windows.net/"
)

destination_path = (
    f"wasbs://{destination_container}@"
    f"{storage_account}.blob.core.windows.net/"
)


# STEP 4: List Files from Source Container

files = dbutils.fs.ls(source_path)

print("Files available in source container:")

for file in files:
    print(file.name)


# STEP 5: Move Excel Files

for file in files:

    if file.name.lower().endswith((".xlsx", ".xls")):

        source_file = file.path

        destination_file = destination_path + file.name

        # Move file from source to destination
        moved = dbutils.fs.mv(
            source_file,
            destination_file
        )

        if moved:
            print("Successfully moved:", file.name)
        else:
            print("Failed to move:", file.name)


# STEP 6: Verify Destination Container

print("\nFiles in destination container:")

destination_files = dbutils.fs.ls(destination_path)

for file in destination_files:
    print(file.name)


print("\nExcel file movement completed!")

In [0]:
pip install azure-storage-blob pandas openpyxl xlrd sqlalchemy pyodbc

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 126.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [openpyxl]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
# Install:
# pip install azure-storage-blob pandas openpyxl xlrd sqlalchemy pyodbc

import pandas as pd
import io
import re
from urllib.parse import quote_plus
from azure.storage.blob import BlobServiceClient
from sqlalchemy import create_engine
from sqlalchemy.types import NVARCHAR

# ==================================================
# 1. AZURE BLOB STORAGE
# ==================================================

storage_connection = (
    "DefaultEndpointsProtocol=https;"
    "AccountName=insurancedataset2026;"
    "AccountKey=YOUR_ACCOUNT_KEY;"
    "EndpointSuffix=core.windows.net"
)

container_name = "destination"

blob_service = BlobServiceClient.from_connection_string(
    storage_connection
)

container = blob_service.get_container_client(container_name)

# ==================================================
# 2. AZURE SQL DATABASE
# ==================================================

server = "insurancedatabase2026.database.windows.net"
database = "insurancedatabase2026"
username = "edukron@insurancedatabase2026"
password = "YOUR_PASSWORD"

connection = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER=tcp:{server},1433;"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect="
    + quote_plus(connection),
    fast_executemany=True
)

# ==================================================
# 3. CLEAN COLUMN NAMES
# ==================================================

def clean_name(name):
    name = re.sub(r"[^a-zA-Z0-9_]", "_", str(name))
    name = name.strip("_").lower()[:110]
    return name or "column"

# ==================================================
# 4. READ EXCEL AND PUSH TO AZURE SQL
# ==================================================

for blob in container.list_blobs():

    if blob.name.lower().endswith((".xlsx", ".xls")):

        try:
            print("Processing:", blob.name)

            # Download Excel file
            blob_client = container.get_blob_client(blob.name)
            data = blob_client.download_blob().readall()

            # Read Excel
            df = pd.read_excel(
                io.BytesIO(data),
                dtype=str
            )

            if df.empty:
                print("Skipping empty file:", blob.name)
                continue

            # Create table name from Excel filename
            filename = blob.name.split("/")[-1]
            table_name = clean_name(filename.rsplit(".", 1)[0])

            # Clean and deduplicate column names
            columns = []
            for col in df.columns:
                base = clean_name(col)
                candidate = base
                i = 1
                while candidate in columns:
                    candidate = f"{base[:100]}_{i}"
                    i += 1
                columns.append(candidate)

            df.columns = columns

            # Handle missing values
            df = df.astype(object).where(pd.notna(df), None)

            # Push directly to Azure SQL Database
            df.to_sql(
                name=table_name,
                con=engine,
                schema="dbo",
                if_exists="replace",
                index=False,
                chunksize=500,
                dtype={
                    col: NVARCHAR(length=None)
                    for col in df.columns
                }
            )

            print(f"SUCCESS: {blob.name} -> dbo.{table_name}")
            print("Rows inserted:", len(df))

        except Exception as e:
            print("FAILED:", blob.name, str(e))

engine.dispose()

print("All Excel files processed!")